<a href="https://colab.research.google.com/github/Aksinhaa/ColabFold/blob/main/TGW_trimming_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



High-throughput sequencing data contains raw reads that are not immediately suitable for analysis. These reads may include sequencing errors, PCR artifacts, and ambiguous alignments. The steps of mapping, deduplication, and filtering are critical to transform raw data into reliable, biologically meaningful information.



Mapping aligns sequencing reads to a reference genome so we can determine where each read originated from.

* Without mapping, reads are just short, uninformative sequences
* Mapping provides genomic context (e.g., gene location, variants)
* Accurate alignment is essential for downstream tasks like variant calling, expression analysis, or peak detection


---


Moreover, during library preparation, PCR amplification can create duplicate reads from the same original DNA fragment. These duplicates do not represent independent biological observations.

* They artificially inflate read counts
* They can bias variant frequency and coverage estimates
* In extreme cases, they may lead to false-positive results

---



In [ ]:
%%bash
# Install Miniconda
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda

# Initialize conda
source /usr/local/miniconda/etc/profile.d/conda.sh

# Accept Terms of Service
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Add channels
conda config --add channels defaults
conda config --add channels bioconda
conda config --add channels conda-forge

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda create -y -n ngs_env bwa samtools qualimap

In [ ]:
%%bash
mkdir -p /content/workshop_data/{reference,trimmed_reads,mapped_reads,aligned_bam}
cd /content/workshop_data
pwd

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/trimmed_reads

wget https://zenodo.org/records/20049847/files/Hoggar1.trim_combined.fq.gz
wget https://zenodo.org/records/20049847/files/Niger.trim_combined.fq.gz
wget https://zenodo.org/records/20049847/files/Senegal-P.trim_combined.fq.gz


This step downloads the reference genome and its index files, which are essential for aligning sequencing reads. During mapping, tools like BWA compare each read to the reference genome to determine its most likely genomic position. Without a reference, reads cannot be placed, and downstream analyses like variant calling would not be possible. Index files are precomputed data structures that allow fast and efficient searching of the reference during alignment.

###  Main Reference File

* **`chr31.fasta`**: The actual DNA sequence of the reference chromosome. All reads are aligned against this sequence.

### Index Files (used by BWA and other tools)

* **`.amb`**: Marks ambiguous bases (e.g., N’s) in the reference.
* **`.ann`**: Stores annotation about sequence names and lengths.
* **`.bwt`**: Core Burrows-Wheeler Transform index used for fast alignment.
* **`.pac`**: Packed version of the reference sequence (compressed format).
* **`.sa`**: Suffix array index that helps locate matches quickly.
* **`.fai`**: FASTA index used for quick random access to specific regions (used by tools like Samtools).
* **`.dict`**: Sequence dictionary file required by some tools (e.g., GATK) for reference metadata.

In summary, the reference genome provides the template for alignment, while the index files make the mapping process fast and computationally efficient.


In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/reference

wget https://zenodo.org/records/19947652/files/chr31.fasta
wget https://zenodo.org/records/19947652/files/chr31.fasta.amb
wget https://zenodo.org/records/19947652/files/chr31.fasta.ann
wget https://zenodo.org/records/19947652/files/chr31.fasta.bwt
wget https://zenodo.org/records/19947652/files/chr31.fasta.dict
wget https://zenodo.org/records/19947652/files/chr31.fasta.fai
wget https://zenodo.org/records/19947652/files/chr31.fasta.pac
wget https://zenodo.org/records/19947652/files/chr31.fasta.sa

This step performs read alignment using BWA, specifically the `bwa aln` algorithm. The loop processes each sample individually, taking the cleaned and combined FASTQ file and aligning it to the reference genome (`chr31.fasta`).

 The output is a `.sai` file (suffix array index), which stores intermediate alignment information and will be used in the next step to generate the final SAM file.

 `-t 2` uses two threads for faster computation, while `-n 0.01` allows a higher mismatch rate to account for sequencing errors and post-mortem damage.

 The `-l 1024` option disables seeding, which improves alignment sensitivity for short and fragmented reads, and `-o 2` allows small gaps (insertions/deletions) during alignment.

Overall, this step maps sequencing reads to their most likely positions in the reference genome, producing alignment coordinates that are essential for downstream processing such as variant calling and quality assessment.


In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Running bwa aln for $sample..."

  bwa aln \
  -t 2 \
  -n 0.01 \
  -l 1024 \
  -o 2 \
  reference/chr31.fasta \
  trimmed_reads/${sample}.trim_combined.fq.gz \
  > mapped_reads/${sample}.sai

done

#**NOTE: If the above step is taking too long, you may stop the run and download the files provided below. These files are shared due to time constraints. Please repeat the steps after the workshop. Before rerunning, delete all partially generated files from the folder section to your left panel, to avoid errors.**

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data/mapped_reads/

wget https://zenodo.org/records/20049847/files/Hoggar1.sai
wget https://zenodo.org/records/20049847/files/Niger.sai
wget https://zenodo.org/records/20049847/files/Senegal-P.sai


Once you have the .sai files with you, we need to convert them into a **SAM** file.
This step uses BWA (bwa samse) to convert the intermediate .sai files into SAM format, which contains the final alignment information. It takes the reference genome, the .sai alignment data, and the original trimmed reads as input. The output .sam file stores detailed information about where each read maps on the reference genome. This format is human-readable and serves as the basis for further processing like sorting and conversion to BAM.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Generating SAM for $sample..."

  bwa samse \
  reference/chr31.fasta \
  mapped_reads/${sample}.sai \
  trimmed_reads/${sample}.trim_combined.fq.gz \
  > mapped_reads/${sample}.sam

done


The SAM files are needed to convert into BAM files for faster processing and savew space in you disk.
This loop iterates over three samples (Hoggar1, Niger, Senegal-P) and processes each one.
For each sample, Samtools sort converts .sam files into sorted .bam files and saves them in the aligned_bam folder.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Sorting $sample..."

  samtools sort \
  -O bam \
  mapped_reads/${sample}.sam \
  -o aligned_bam/${sample}.bam

done

For each BAM file, Samtools index creates a .bai index file to enable fast data access.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Indexing $sample..."

  samtools index aligned_bam/${sample}.bam

done

This step filters aligned reads using Samtools to retain only high-quality and reliable mappings. The `-F 4 `option removes unmapped reads, while `-q 30` keeps reads with high mapping quality, reducing false alignments. The `-h ` flag preserves the header, and `-b` outputs the result in compressed BAM format. This filtering is important to ensure that downstream analyses, such as variant calling, are based only on confident and biologically meaningful data.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Filtering $sample..."

  samtools view \
  -h \
  -F 4 \
  -q 30 \
  -b \
  aligned_bam/${sample}.bam \
  > aligned_bam/${sample}.filtered.bam

done

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Indexing filtered $sample..."

  samtools index aligned_bam/${sample}.filtered.bam

done

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Removing duplicates for $sample..."

  samtools markdup \
  -r \
  -s \
  aligned_bam/${sample}.filtered.bam \
  aligned_bam/${sample}.filtered.dedup.bam

done

This step removes duplicate reads using Samtools (markdup), which is important for reducing biases introduced during PCR amplification. If not removed, these duplicates can inflate coverage and lead to incorrect variant calling.

In the script, a loop processes each sample and applies samtools markdup to the filtered BAM file. The `-r` flag removes duplicate reads rather than just marking them, ensuring they are excluded from downstream analysis. The `-s` option provides statistics about duplicate removal, which helps assess the extent of duplication in the dataset.

This step is crucial for ensuring that each DNA fragment is represented only once, improving the accuracy of SNP calling and downstream population analyses.

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/workshop_data

for sample in Hoggar1 Niger Senegal-P
do
  echo "Indexing deduplicated $sample..."

  samtools index aligned_bam/${sample}.filtered.dedup.bam

done